# 06 · Make WVD sensitivity tables by length, frequency and quantization

**Depends on 05.** Calls the **unchanged original `33_v6_binned_validation_hexbins.py`**. It creates station-matched V6/CHMI pairs, frequency/length/sensitivity figures, `binned_validation_metrics.csv` and `population_counts.csv` for the GRAF (CML-only) branch.

Important: the 7 core NetCDFs do **not** contain the V6 diagnostics and link quantization series CSV required here. Configure `extras.wvd_v6_diagnostics_csv` and `extras.wvd_quantization_csv`. Those supporting inputs can be provided with a reproducibility supplement; they cannot be replaced with an invented quantization estimate.

In [1]:
from pathlib import Path
import sys, json
# One shared configuration for ALL notebooks (no per-notebook path editing).
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'tools_verify_originals.py').is_file(), f'Wrong notebook working folder: {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
from essd_cz.connected import Project
SETTINGS=json.loads((ROOT/'config/active_profile.json').read_text(encoding='utf-8'))
PROFILE=SETTINGS.get('profile','hpc')
if PROFILE not in ('hpc','user'):raise ValueError("Profile must be 'hpc' or 'user'")
CONFIG=ROOT/('config/hpc_source.example.json' if PROFILE=='hpc' else 'config/user_paths.json')
WORKDIR=ROOT/SETTINGS.get('workspace','results/connected_workflow')
project=Project(CONFIG,WORKDIR)
print('Profile:',PROFILE,'Data config:',project.config,'Output:',project.output)


Profile: hpc Data config: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/config/hpc_source.example.json Output: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow


### NetCDF4 compatibility fix (only data access)

The original sensitivity script may raise `IndexError` when xarray's lazy NetCDF4 backend combines two scalar indexes. This notebook writes a **temporary compatibility copy of the unchanged original script** inside the output workspace. Only the two method/series reads are changed to load a **single sliced time series**, preserving the exact values, statuses, station matching, bin definitions and statistics. The original under `original/code/wvd/` is never modified.

The optional check below also detects a separate problem: Notebook 05 may have generated a **small demonstration V6 file** while the configured diagnostics and quantization CSV describe a larger population. For a paper-population analysis those inputs must refer to the same series. It will not silently remove links.

In [3]:
import xarray as xr
import pandas as pd

v6_file = project.get("wvd_v6")
diagnostics = pd.read_csv(
    project.extras["wvd_v6_diagnostics_csv"]
)
quantization = pd.read_csv(
    project.extras["wvd_quantization_csv"]
)

with xr.open_dataset(v6_file) as ds:
    n = ds.sizes["series"]

print("V6 series:", n)
print("Diagnostics rows:", len(diagnostics))
print("Quantization rows:", len(quantization))

print("Diagnostics series indices:",
      diagnostics.series_index.nunique())
print("Quantization series indices:",
      quantization.series_index.nunique())

V6 series: 3
Diagnostics rows: 1352
Quantization rows: 1352
Diagnostics series indices: 1352
Quantization series indices: 1352


In [2]:
# Original V6 sensitivity, with a traceable NetCDF4 indexing compatibility copy.
from pathlib import Path
import ast, hashlib, json, subprocess, sys
import pandas as pd
import xarray as xr

BRANCH = 'GRAF'
MAX_STATION_DISTANCE_KM = 5.0
DPI = 200
RUN_SENSITIVITY = True  # False = dependency/index audit only

v6_file = project.get('wvd_v6')  # mandatory hand-off from Notebook 05
quant = project.extras.get('wvd_quantization_csv')
diag = project.extras.get('wvd_v6_diagnostics_csv')
original = ROOT / 'original/code/wvd/33_v6_binned_validation_hexbins.py'
helper = ROOT / 'original/code/wvd/21_validate_wvd_v6_vs_original.py'
needed = {'V6 NetCDF': v6_file, 'quantization CSV': quant,
          'diagnostics CSV': diag, 'original analysis': original,
          'original validation helper': helper}
missing = [name for name, path in needed.items() if path is None or not Path(path).is_file()]
if missing:
    raise FileNotFoundError(f'Missing required WVD inputs: {missing}')

with xr.open_dataset(v6_file, cache=False) as ds:
    n_series = int(ds.sizes['series'])
    for name in ('wvd_internal_anchor', 'retrieval_status'):
        if ds[name].dims != ('method', 'series', 'time'):
            raise ValueError(f'{name}: unexpected dimensions {ds[name].dims}')
    print('V6 directional series:', n_series)

# Audit the population; do NOT silently filter series or change the validation sample.
qmeta = pd.read_csv(quant, nrows=0)
index_source = quant if 'series_index' in qmeta.columns else diag
index_df = pd.read_csv(index_source, usecols=['series_index'])
series_idx = pd.to_numeric(index_df['series_index'], errors='raise')
if series_idx.isna().any() or (series_idx < 0).any() or (series_idx >= n_series).any():
    bad = int(((series_idx < 0) | (series_idx >= n_series) | series_idx.isna()).sum())
    raise RuntimeError(
        f'V6 NetCDF has {n_series} series, but {bad} metadata rows refer to unavailable '
        'series indices. Notebook 05 probably produced a small demonstration file, while '
        'the quantization/diagnostics CSVs describe the full population. Supply matching '
        'V6 and CSV inputs for the paper analysis; do not silently drop those links.'
    )
print('V6-series index range is compatible with the metadata:', int(series_idx.min()), int(series_idx.max()))

source = original.read_text(encoding='utf-8')
substitutions = {
    'v6["wvd_internal_anchor"].isel(method=mi, series=vi).values':
        'v6["wvd_internal_anchor"].isel(method=slice(mi, mi + 1), series=slice(vi, vi + 1)).load().values[0, 0, :]',
    'v6["retrieval_status"].isel(method=mi, series=vi).values':
        'v6["retrieval_status"].isel(method=slice(mi, mi + 1), series=slice(vi, vi + 1)).load().values[0, 0, :]',
}
for old, new in substitutions.items():
    if source.count(old) != 1:
        raise RuntimeError(f'Unexpected source revision; refusing automated indexing patch: {old}')
    source = source.replace(old, new, 1)
ast.parse(source)  # syntax check before any execution
out = project.stage_dir('WVD_V6_QUANTIZATION_BINNED_VALIDATION')
compat_script = out / '33_v6_binned_validation_hexbins_NETCDF4_COMPAT.py'
compat_script.write_text(source, encoding='utf-8')
original_sha = hashlib.sha256(original.read_bytes()).hexdigest()
compat_sha = hashlib.sha256(compat_script.read_bytes()).hexdigest()
print('Original scientific script retained:', original)
print('Read-only indexing compatibility copy:', compat_script)

candidate = (SETTINGS.get('science_python_by_profile') or {}).get(PROFILE)
science_python = Path(candidate).expanduser() if candidate else Path(sys.executable)
if not science_python.is_file():
    raise FileNotFoundError(f'Configured science Python absent: {science_python}')

if RUN_SENSITIVITY:
    cmd = [str(science_python), str(compat_script),
           '--root', str(project.output), '--v6-nc', str(v6_file),
           '--v6-diagnostics-csv', str(diag), '--quant-csv', str(quant),
           '--raw-cml-nc', str(project.paths['raw']),
           '--humidity-nc', str(project.paths['humidity']),
           '--temperature-nc', str(project.paths['temperature']),
           '--validation-helper-script', str(helper),
           '--station-assignments-csv', str(out/'station_assignments.csv'),
           '--out-dir', str(out), '--branch', BRANCH,
           '--max-station-distance-km', str(MAX_STATION_DISTANCE_KM),
           '--dpi', str(DPI)]
    # No --reuse-pairs: rerun matching from the actual inputs before plotting.
    print('Running indexing-compatibility copy:', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    # Register only outputs that a successful calculation actually wrote.
    project.register('wvd_binned_metrics', out/'binned_validation_metrics.csv',
                     'original 33 WVD algorithm; 2 NetCDF single-series read adjustments',
                     branch=BRANCH, original_sha256=original_sha, compat_sha256=compat_sha)
    project.register('wvd_link_population', out/'validation_population_links.csv',
                     'original 33 station matching; NetCDF read compatibility only')
    (out/'indexing_compat_provenance.json').write_text(json.dumps({
        'original_script': str(original), 'original_sha256': original_sha,
        'compatibility_script': str(compat_script), 'compatibility_sha256': compat_sha,
        'only_changes': list(substitutions), 'v6_nc': str(v6_file),
        'quantization_csv': str(quant), 'diagnostics_csv': str(diag),
        'branch': BRANCH, 'n_v6_series': n_series,
        'python': str(science_python),
    }, indent=2)+'\n', encoding='utf-8')
    print('WVD binned sensitivity outputs:', out)
else:
    print('PREFLIGHT ONLY — no WVD sensitivity run was started')

V6 directional series: 3


RuntimeError: V6 NetCDF has 3 series, but 1349 metadata rows refer to unavailable series indices. Notebook 05 probably produced a small demonstration file, while the quantization/diagnostics CSVs describe the full population. Supply matching V6 and CSV inputs for the paper analysis; do not silently drop those links.